# Gradient Boosting — Performance with Discipline (and Leakage Avoidance)

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026F_predictive_analytics_QM474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2027S_predictive_analytics_QM474/blob/main/notebooks/nb13_gradient_boosting_student.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain how gradient boosting reduces **bias** by training trees sequentially on the residuals of the previous trees, on both classification and regression cases.
2. Fit `GradientBoostingClassifier` and `GradientBoostingRegressor` and test, with `compare_paired` on identical folds, whether boosting earns a **worthwhile gain** over each case's current champion from nb12.
3. Diagnose the `learning_rate × n_estimators` trade-off — why the two must be tuned together — and treat a tuning grid's best cell as a **nominee** that must beat the default GBM before it replaces it.
4. Climb the full **champion ladder** (linear reference → tree → forest → GBM), check whether the champion is robust across the margin band, and state the switching cutoff for the closest rival.
5. Recognize and prevent **boosting's amplification of leaky features** — the failure mode that nb09's leakage case studies set up.

---


> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises** — one per case. Complete both before submitting your notebook.

---

## 💼 Why This Matters

Random forests reduce **variance** by averaging many trees. Gradient boosting reduces **bias** by stacking trees sequentially — each new tree fits the *residuals* (regression) or the gradient of the log-loss (classification) of the running ensemble. The two strategies attack different parts of the bias/variance decomposition, which is why nb14's selection ceremony puts both on the same roster.

For the **State Health Department's** screening pipeline, the question is the one nb12 already asked of the forest: *"can a tree ensemble rank biopsies better than the logistic model we can explain?"* After nb12 the review board's champion is still `LogReg(C=1.0)`, and two of its five fold scores are already 0.9995 — so on this evidence there is very little ranking headroom left for any challenger to show.

For **HomeValue Analytics'** price-prediction model, the current champion after nb12 is the **default random forest**, which earned a worthwhile gain over OLS. Boosting has to clear the same bar against the forest: a corrected interval of the per-fold gain whose lower end sits above the CFO's base margin of **0.01 R²**. Nothing in this notebook assumes it will — the comparison decides.

The cost of boosting is real: it is sequential (it cannot parallelize across trees), more sensitive to hyperparameters than random forests, and prone to overfitting if you let it run too many iterations. Two of these risks have specific mitigations covered in this notebook: tune `learning_rate × n_estimators` together, and **never let gradient boosting see a leaky feature** — boosting will amplify the leak more aggressively than any other algorithm in this course.

> **A question that often comes up here:** *"If boosting is often strong on tabular data, why not skip forests entirely?"* Three answers. First, forests give you OOB estimates and parallel fits — properties boosting lacks. Second, the four-method importance table from nb12 transfers to boosting, but the *interpretation* of MDI is harder for boosting (sequential trees accumulate importance differently). Third, the right answer is never "always boost" but "score both on the identical folds, put the corrected interval around the per-fold gain, and keep the current champion unless that interval clears the margin you declared up front." Today's job is to add gradient boosting to the candidate roster — not to crown it.

---


## 1. Setup — Imports, References, Helpers

Same toolkit as nb11 and nb12: the two nb09 reference pipelines, nb08's `compare_paired` and margin bands (`DELTAS_AUC`, `DELTAS_R2`), and the helpers that print and plot a comparison (`show_comparison`, `fold_table`, `plot_fold_scores`, `plot_gain_intervals`).


> 💡 **Gemini Prompt:** "Set up imports for sklearn GradientBoostingClassifier, GradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor, DecisionTreeClassifier, DecisionTreeRegressor, LogisticRegression, LinearRegression, train_test_split, cross_val_score, StratifiedKFold, KFold, load_breast_cancer, fetch_california_housing, StandardScaler, Pipeline and scipy.stats. Set RANDOM_SEED = 474. Define reference_clf and reference_reg as the nb09 baseline pipelines. Paste the course's compare_paired(scores_a, scores_b, margin) function from nb08 exactly as written, define DELTAS_AUC and DELTAS_R2 as {'small': 0.005, 'base': 0.01, 'large': 0.02}, and define the comparison helpers show_comparison, fold_table, plot_fold_scores and plot_gain_intervals."
>
> **After running, verify:**
> - [ ] `RANDOM_SEED = 474`, `reference_clf`, `reference_reg` defined
> - [ ] Both margin bands printed back, with `base = 0.01`
> - [ ] No import errors


In [ ]:
# Setup — imports, seed, nb09 references, plot helpers, the paired comparison and the margin band
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.base import clone
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import (train_test_split, cross_val_score, StratifiedKFold, KFold,
                                     RandomizedSearchCV)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              GradientBoostingClassifier, GradientBoostingRegressor)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.precision', 4)
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)

CLF_COLOR = '#1f77b4'
REG_COLOR = '#ff7f0e'
GREY      = '#999999'

# --- nb09 reference models: the incumbents every challenger has faced since nb11 ---
reference_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(C=1.0, random_state=RANDOM_SEED, max_iter=5000))
])
reference_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('reg',    LinearRegression())
])

# --- compare_paired: the course's model-comparison rule from nb08 (pasted verbatim) ---
# A is the challenger, B the current champion. Both score arrays must come from ONE splitter object.
def compare_paired(scores_a, scores_b, margin):
    """Paired comparison on identical folds; higher score = better."""
    d = np.asarray(scores_a) - np.asarray(scores_b)
    k = len(d)
    t = stats.t.ppf(0.975, k - 1)
    se = d.std(ddof=1) * np.sqrt(1/k + 1/(k - 1))   # Nadeau–Bengio correction
    lo, hi = d.mean() - t*se, d.mean() + t*se
    if lo > margin:
        verdict = "A is better by a worthwhile amount"
    elif hi < -margin:
        verdict = "B is better by a worthwhile amount"
    elif -margin < lo and hi < margin:
        verdict = "Practically equivalent: choose the simpler model"
    else:
        verdict = "Unresolved: apply the default you declared"
    return d, d.mean(), (lo, hi), verdict

# --- The margin band each stakeholder declared in nb08, before any score was computed ---
# base decides the champion; small and large are the sensitivity check (same numbers through nb14).
DELTAS_AUC = {'small': 0.005, 'base': 0.01, 'large': 0.02}   # MedScreen review board — ROC-AUC
DELTAS_R2  = {'small': 0.005, 'base': 0.01, 'large': 0.02}   # HomeValue CFO — R²
DELTA_AUC  = DELTAS_AUC['base']
DELTA_R2   = DELTAS_R2['base']

def show_comparison(challenger, champion, scores_a, scores_b, deltas, metric):
    """Print one paired comparison: fold gains (description), corrected interval, verdict, band check."""
    d, gain, (lo, hi), verdict = compare_paired(scores_a, scores_b, deltas['base'])
    print(f"{challenger} (A, challenger) vs {champion} (B, current champion) — {metric}")
    print("  per-fold gains A − B (description only): " + ', '.join(f"{x:+.4f}" for x in d))
    print(f"  mean gain {gain:+.4f}; interval corrected for fold overlap [{lo:+.4f}, {hi:+.4f}]")
    print(f"  base margin δ = {deltas['base']}: {verdict}")
    band = {name: compare_paired(scores_a, scores_b, m)[3] for name, m in deltas.items()}
    same = len(set(v.startswith('A is better') for v in band.values())) == 1
    print("  band check: " + '; '.join(f"{n} {deltas[n]} → {v.split(':')[0]}" for n, v in band.items())
          + ("  → robust across the band" if same else "  → margin-sensitive"))
    return d, gain, (lo, hi), verdict

# --- Fold-score summary for ONE model: the five fold scores, their mean and their SD (no interval) ---
def fold_table(score_dict):
    rows = []
    for name, s in score_dict.items():
        s = np.asarray(s)
        rows.append({'model': name, **{f'fold {i+1}': v for i, v in enumerate(s)},
                     'mean': s.mean(), 'sd': s.std(ddof=1)})
    return pd.DataFrame(rows)

def plot_fold_scores(ax, score_dict, ylabel, title, colors=None):
    """Dot strip: each model's five fold scores and their mean — no error bars, no interval."""
    for j, (name, s) in enumerate(score_dict.items()):
        c = (colors or {}).get(name, GREY)
        ax.scatter(np.full(len(s), j) + np.linspace(-0.08, 0.08, len(s)), s, color=c, s=45,
                   edgecolor='black', zorder=3)
        ax.hlines(np.mean(s), j - 0.25, j + 0.25, color='black', linewidth=2, zorder=4)
    ax.set_xticks(range(len(score_dict)))
    ax.set_xticklabels(list(score_dict), fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

def plot_gain_intervals(ax, rows, margin, xlabel, title):
    """rows: list of (label, d, (lo, hi)). Dots = per-fold gains; bar = corrected interval; shaded = ±δ."""
    ax.axvspan(-margin, margin, color=GREY, alpha=0.15, label=f'±δ = ±{margin}')
    ax.axvline(0, color='black', linewidth=1)
    for j, (label, d, (lo, hi)) in enumerate(rows):
        ax.hlines(j, lo, hi, color='black', linewidth=3)
        ax.scatter(d, np.full(len(d), j), color=REG_COLOR if 'R²' in xlabel else CLF_COLOR,
                   s=35, edgecolor='black', zorder=3)
        ax.scatter([np.mean(d)], [j], marker='D', color='white', edgecolor='black', s=60, zorder=4)
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels([r[0] for r in rows], fontsize=9)
    ax.set_xlabel(xlabel)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(True, alpha=0.3, axis='x')

print(f"✓ RANDOM_SEED = {RANDOM_SEED}")
print("✓ nb09 references: reference_clf, reference_reg")
print(f"✓ Margin band (base decides): ROC-AUC {DELTAS_AUC}, R² {DELTAS_R2}")
print("✓ Helpers: compare_paired, show_comparison, fold_table, plot_fold_scores, plot_gain_intervals")


---

## 2. Load Both Datasets

Same discipline as nb11 and nb12: each case is split **once**, 80% training / 20% test, with `random_state=RANDOM_SEED`, so the partition is identical to nb11–nb12. Every fit, every diagnostic, and every tuning decision between here and nb14 lives on the training rows and runs through 5-fold CV; there is no validation set, and the test envelopes stay sealed.

The section is short on purpose — its job is to make `X_train_clf`, `X_train_reg`, `cv_clf`, and `cv_reg` available to every section that follows, so the fold scores you compute today line up fold by fold with nb11's and nb12's.


> 💡 **Gemini Prompt:** "Apply the course's 80/20 split to both datasets. Load `load_breast_cancer(as_frame=True)` and `fetch_california_housing(as_frame=True)`. For each, call `train_test_split` once with `test_size=0.20`, `random_state=474`, and `stratify=y` for classification only, producing `X_train_*` / `X_test_*`. Build `cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=474)` and `cv_reg = KFold(n_splits=5, shuffle=True, random_state=474)`. Print the per-case row counts and a `[LOCKED until nb14]` marker on the test counts."
>
> **After running, verify:**
> - [ ] Classification: train 455, test 114 (out of 569)
> - [ ] Regression: train 16,512, test 4,128 (out of 20,640)
> - [ ] One split per case, no validation set, both splitters with 5 folds
> - [ ] Both test-set lines include `[LOCKED until nb14]`


In [ ]:
# One 80/20 split per case — the same first split as nb08, nb09, nb11 and nb12 under RANDOM_SEED.
data_clf = load_breast_cancer(as_frame=True)
X_clf, y_clf = data_clf.data, data_clf.target
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=RANDOM_SEED, stratify=y_clf
)
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

data_reg = fetch_california_housing(as_frame=True)
X_reg, y_reg = data_reg.data, data_reg.target
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_SEED
)
cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

print(f'Classification — Train: {len(X_train_clf):>5} | Test: {len(X_test_clf):>4} [LOCKED until nb14]')
print(f'Regression     — Train: {len(X_train_reg):>5} | Test: {len(X_test_reg):>4} [LOCKED until nb14]')


**Reading the output:**

One split per case, two locked envelopes, two CV plans — the same as nb11 and nb12. The classification training set has **455 patients** and the regression training set **16,512 tracts** (a 36× size gap that has shaped every modelling decision since nb11). `StratifiedKFold` preserves the \~63% benign / \~37% malignant ratio on the classification side; plain `KFold` works on the regression side because the target is continuous. The `X_test_*` sets stay sealed until nb14's ceremony.

> **A question that often comes up here:** *"My Section 4 numbers for the reference and the forest look exactly like nb12's — is that a problem?"* No — it is the point. nb11, nb12, nb13 and nb14 all use the same `RANDOM_SEED=474`, the same single 80/20 split and the same 5-fold splitters, so they operate on **identical** training rows and identical folds. The nb09 reference and the default forest in Section 4 reproduce nb12's fold scores to four decimal places, which is what lets you put a boosted model against them fold by fold. That consistency is the discipline that makes nb14's selection ceremony defensible.

**Key takeaway.** Same split, same seed, same CV folds across nb11 → nb14. Every fold score in nb13 pairs directly with the nb11 and nb12 fold scores, and the test envelopes stay sealed until nb14.

---


## 3. Boosting vs Bagging — Sequential vs Parallel

nb12 built up the **bagging** recipe: train many trees in parallel on bootstrap samples, then average their predictions. Today's gradient boosting uses the opposite recipe — **boosting**: train one tree, look at where the ensemble is still wrong, train the next tree specifically to fix those mistakes, and keep going. The schematic below contrasts the two side-by-side, because the *shape* of the algorithm matters before any code runs.

Think of the contrast in HomeValue's terms. Bagging is **a panel of independent analysts** — each one estimates a neighborhood's median value from a slightly different set of comparable sales, and the firm publishes the average across the panel. The analysts do not know about each other; the averaging is what cancels out individual noise. Boosting is **an apprentice-and-master chain** — analyst 1 hands in an estimate for every neighborhood, analyst 2 looks at where 1 was wrong and adds a small correction, analyst 3 looks at where 1 and 2 *together* are still wrong and adds another correction, and so on for hundreds of iterations. Each new analyst is *deliberately specialized* on the running team's leftover error. That sequential structure is what makes boosting reduce **bias** (systematic mistakes the ensemble keeps making) rather than variance.

> 💡 **Gemini Prompt:** "Render a side-by-side schematic comparing bagging (left) and boosting (right). Bagging panel: draw five parallel arrows from a central training-data block, each ending at a tree box, with a final averaging step on the right labeled 'Average'. Boosting panel: draw a chain — Tree 1 fits the data, an arrow flows from Tree 1 to a 'Residuals' box, then to Tree 2, then to another 'Residuals' box, then to Tree 3, ending with a final 'Weighted sum' box. Label the left panel 'Bagging — parallel, order-independent (variance ↓)' and the right panel 'Boosting — sequential, order-dependent (bias ↓)'. Use neutral grey for the tree boxes; no need for axes."
>
> **After running, verify:**
> - [ ] Left panel shows parallel trees feeding into an average box
> - [ ] Right panel shows trees chained through residuals to a final sum
> - [ ] Subtitles under each panel name the variance-vs-bias distinction


In [ ]:
# Schematic: bagging vs boosting
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bagging — parallel fan
ax = axes[0]
for i in range(5):
    ax.annotate('', xy=(0.55, 0.5), xytext=(0.15, 0.85 - i*0.15),
                arrowprops=dict(arrowstyle='->', color=CLF_COLOR, lw=2))
    ax.text(0.10, 0.85 - i*0.15, f'Tree {i+1}', ha='right', va='center', fontsize=11)
ax.text(0.55, 0.5, 'Average\n(or vote)', ha='center', va='center', fontsize=12,
        bbox=dict(boxstyle='round', facecolor=GREY, alpha=0.3))
ax.text(0.85, 0.5, 'Prediction', ha='center', va='center', fontsize=11,
        bbox=dict(boxstyle='round', facecolor=CLF_COLOR, alpha=0.3))
ax.annotate('', xy=(0.95, 0.5), xytext=(0.65, 0.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.set_title('Bagging (Random Forest) — parallel, independent trees', fontsize=12, fontweight='bold')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

# Boosting — sequential chain
ax = axes[1]
for i in range(5):
    x_pos = 0.10 + i*0.18
    ax.text(x_pos, 0.5, f'T{i+1}', ha='center', va='center', fontsize=11,
            bbox=dict(boxstyle='round', facecolor=REG_COLOR, alpha=0.3))
    if i < 4:
        ax.annotate('', xy=(x_pos + 0.13, 0.5), xytext=(x_pos + 0.05, 0.5),
                    arrowprops=dict(arrowstyle='->', color='black', lw=2))
    ax.text(x_pos, 0.30, 'fits\nresiduals\nof T1..T'+str(i) if i > 0 else 'fits\nfull data',
            ha='center', va='center', fontsize=8, color=GREY)
ax.text(0.5, 0.78, 'Σ scaled by learning_rate',
        ha='center', va='center', fontsize=11,
        bbox=dict(boxstyle='round', facecolor=GREY, alpha=0.3))
ax.set_title('Boosting (Gradient Boosting) — sequential, each tree fixes the previous error',
             fontsize=12, fontweight='bold')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

plt.tight_layout()
plt.show()


**Reading the output:**

The two diagrams capture the structural difference. Bagging is **parallel and order-independent** — train tree 1 and tree 5 simultaneously, average them at the end, and the order does not matter. Boosting is **sequential and order-dependent** — tree 5 is fit on residuals that trees 1 through 4 produced together, so the order matters and the trees cannot be trained in parallel.

The order dependence is what makes boosting *slower per training run* than a forest of the same size — you cannot parallelize across trees on a multi-core machine, so the wall-clock cost scales linearly with `n_estimators`. It is also why boosting is often strong on tabular data, because each new tree's job is **specifically defined** (*"fix what the running ensemble still gets wrong"*) rather than the bagging analyst's job of *"be a different tree."* The price you pay for that focused specialization is sensitivity to hyperparameters — Sections 5 and 6 walk through the two main dials (`learning_rate`, `n_estimators`) and how they interact.

> **A question that often comes up here:** *"if boosting reduces bias and bagging reduces variance, why not always pick boosting?"* Three reasons. First, **sequential dependence**: you cannot fit `n_estimators` boosted trees in parallel on a multi-core machine, so wall-clock training time scales linearly with `n_estimators` while a random forest of the same size fits in roughly `O(1)` wall-clock time given enough cores. Second, **hyperparameter sensitivity**: a random forest works well at sklearn's defaults, but a GBM at the wrong `learning_rate × n_estimators × max_depth` combination can underfit or overfit dramatically — Section 6's joint-tuning grid is the antidote. Third, **overfitting risk**: more trees never hurt a forest, but more trees in a GBM *can* and *do* overfit if `n_estimators` is set too high relative to the dataset's signal. The right framing for nb14: bagging and boosting are complementary, and what decides which one ships per business case is the paired comparison — score both on the identical folds, subtract fold by fold, put the corrected interval around the mean gain, and keep the current champion unless that interval clears the margin the stakeholder declared in nb08.

**Key takeaway.** Bagging cancels variance by averaging independent trees; boosting reduces bias by stacking trees that each correct the previous error. The two are complementary — nb14 will compare both, under identical CV folds, alongside the linear references from nb09. Neither is universally better; the data decides.

---


## 4. Baseline Gradient Boosting — Default Hyperparameters on Both Cases

Before you tune anything, run the GBM at scikit-learn's defaults and ask the only question that matters: **does it earn a worthwhile gain over each stakeholder's current champion?** Those champions are nb12's two verdicts:

- **Classification (Wisconsin Breast Cancer):** the nb09 reference `LogReg(C=1.0)` — still the State Health Department's champion after nb12, because neither the tree nor the forest earned a worthwhile gain over it.
- **Regression (California Housing):** the **default random forest**, `RandomForestRegressor(random_state=474)` — HomeValue's champion after nb12, where it earned a worthwhile gain over OLS.

The scikit-learn defaults for `GradientBoostingClassifier` and `GradientBoostingRegressor` are:

- `n_estimators=100` — one hundred boosting iterations
- `learning_rate=0.1` — each tree's contribution is shrunk by 0.1 before being added to the running ensemble
- `max_depth=3` — each tree is intentionally shallow (the *weak learner* idea)
- `loss='log_loss'` for classification, `'squared_error'` for regression

The output has the usual two layers. The **description** is a table and a dot strip of each model's five fold scores, mean and SD — no error bars around a single model. The **decision** is `compare_paired(default GBM, current champion, δ)` on the identical folds, read against the margin band both stakeholders declared in nb08: **base 0.01** decides (ROC-AUC for the review board, R² for HomeValue's CFO), and 0.005 / 0.02 check the sensitivity. Only *A is better by a worthwhile amount* would put the default GBM in the champion's place.


> 💡 **Gemini Prompt:** "Score three candidates per case on identical 5-fold folds. **Classification:** reference_clf (the LogReg(C=1.0) pipeline), RandomForestClassifier(random_state=474) — the forest nb12 carried forward — and GradientBoostingClassifier(random_state=474) at defaults, with cv_clf and scoring='roc_auc'. **Regression:** reference_reg (the OLS pipeline), RandomForestRegressor(random_state=474) and GradientBoostingRegressor(random_state=474), with cv_reg and scoring='r2'. Print each case's fold_table (five fold scores, mean, SD). Then run show_comparison with the default GBM as A against each case's current champion as B — LogReg on classification, the forest on regression — using DELTAS_AUC / DELTAS_R2, and plot the fold scores and the paired gain intervals."
>
> **After running, verify:**
> - [ ] Every candidate on a case uses the same `cv_clf` / `cv_reg` object
> - [ ] Tables show five fold scores, mean and SD — no interval around a single model
> - [ ] Each case prints per-fold gains, the corrected interval and one of the four verdicts at δ = 0.01
> - [ ] The reference and forest fold scores match nb12's


In [ ]:
# Default GBM vs each case's current champion (named at the end of nb12) — identical 5-fold folds.
# The forest nb12 carried forward is scikit-learn's default forest on both cases (nb12 §5).
gbm_clf_default    = GradientBoostingClassifier(random_state=RANDOM_SEED)
gbm_reg_default    = GradientBoostingRegressor(random_state=RANDOM_SEED)
forest_clf_carried = RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1)
forest_reg_carried = RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1)

clf_scores = {
    'nb09 reference: LogReg(C=1.0)':  cross_val_score(reference_clf,      X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
    'Random Forest (default, nb12)':  cross_val_score(forest_clf_carried, X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
    'Gradient Boosting (defaults)':   cross_val_score(gbm_clf_default,    X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
}
reg_scores = {
    'nb09 reference: OLS':            cross_val_score(reference_reg,      X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1),
    'Random Forest (default, nb12)':  cross_val_score(forest_reg_carried, X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1),
    'Gradient Boosting (defaults)':   cross_val_score(gbm_reg_default,    X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1),
}

print("=== CLASSIFICATION — 5-fold CV ROC-AUC: fold scores, mean, SD ===")
print(fold_table(clf_scores).to_string(index=False))
print()
print("=== REGRESSION — 5-fold CV R²: fold scores, mean, SD ===")
print(fold_table(reg_scores).to_string(index=False))

# The decision: default GBM (A, challenger) vs each case's current champion (B)
print()
d_clf, g_clf, ci_clf, v_clf = show_comparison('Gradient Boosting (defaults)', 'LogReg(C=1.0) — MedScreen champion',
                                              clf_scores['Gradient Boosting (defaults)'], clf_scores['nb09 reference: LogReg(C=1.0)'],
                                              DELTAS_AUC, 'ROC-AUC')
print()
d_reg, g_reg, ci_reg, v_reg = show_comparison('Gradient Boosting (defaults)', 'Random Forest (default) — HomeValue champion',
                                              reg_scores['Gradient Boosting (defaults)'], reg_scores['Random Forest (default, nb12)'],
                                              DELTAS_R2, 'R²')

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
plot_fold_scores(axes[0, 0], clf_scores, '5-fold CV ROC-AUC', 'Classification — fold scores (bar = mean)',
                 colors={'Gradient Boosting (defaults)': CLF_COLOR})
plot_fold_scores(axes[0, 1], reg_scores, '5-fold CV R²', 'Regression — fold scores (bar = mean)',
                 colors={'Gradient Boosting (defaults)': REG_COLOR})
plot_gain_intervals(axes[1, 0], [('default GBM − LogReg', d_clf, ci_clf)], DELTA_AUC,
                    'Gain in ROC-AUC', 'MedScreen — default GBM vs the champion')
plot_gain_intervals(axes[1, 1], [('default GBM − forest', d_reg, ci_reg)], DELTA_R2,
                    'Gain in R²', 'HomeValue — default GBM vs the champion')
fig.suptitle('Default GBM vs the current champions — has gradient boosting earned a place at its defaults?',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


**Reading the output:**

The two business cases give the same answer — not yet — for different reasons.

**Classification case (Wisconsin Breast Cancer).** LogReg(C=1.0) averages a CV ROC-AUC of **0.994** (SD 0.007), the default forest **0.988** (SD 0.014) and the default GBM **0.987** (SD 0.015). The default GBM's per-fold gains over LogReg are −0.008, −0.002, −0.004, −0.002 and −0.019: a mean of **−0.007** with a corrected interval of **[−0.021, +0.007]**. The interval crosses both −0.01 and zero and stays below +0.01: ***unresolved*** at every margin in the band. Unresolved means the declared default — **the review board keeps `LogReg(C=1.0)`**. Notice how little room there is: two of LogReg's folds are already at 0.9995, so on those folds no challenger can gain more than about 0.0005.

**Regression case (California Housing).** The default forest averages a CV R² of **0.804** (SD 0.012) and the default GBM **0.790** (SD 0.013); OLS sits far below at 0.604. The default GBM trails the forest on all five folds, by −0.017, −0.009, −0.014, −0.008 and −0.020 — a mean of **−0.014** with a corrected interval of **[−0.023, −0.004]**. The interval lies entirely below zero, so the GBM is behind; but its upper end (−0.004) does not reach below −0.01, so the evidence cannot say the forest is better *by a worthwhile amount*. The verdict is ***unresolved***, and either way **HomeValue keeps the forest**. The mechanical reason the default GBM falls short is its budget: at `learning_rate=0.1 × n_estimators=100` the shallow boosted ensemble has not yet accumulated enough total correction to match a hundred fully grown bagged trees. Sections 5 and 6 explore that budget.

> **A question that often comes up here:** *"If the default GBM does not beat either champion, why was it worth fitting?"* Two reasons. First, **the comparison itself is the discipline**: every new candidate faces the current champion on the paired per-fold gain, read against the margin declared in advance — and a default GBM that sits about 0.014 R² behind the forest tells you that boosting is *worth tuning* but not worth shipping out of the box. Second, **the default GBM is the family's starting point**: Section 6's tuning grid produces a nominee, and that nominee has to beat exactly this default before it can face the forest.

**Key takeaway.** At its defaults, gradient boosting earns nothing over either champion: unresolved against LogReg on classification (mean gain −0.007), and unresolved — behind on every fold, but not by a worthwhile amount — against the default forest on regression (mean gain −0.014). The default GBM is the *floor* for the tuning work in Sections 5 and 6.

---


## 5. Learning Rate Trade-off — Slow Learning Beats Fast Learning

Think of `learning_rate` as a **volume knob on each tree's vote**. With `learning_rate=0.1` and 100 trees, every tree casts a 10% vote and the ensemble averages 100 quiet voices into one prediction. With `learning_rate=0.5` and 100 trees, every tree casts a 50% vote — the ensemble converges in a hurry but is more likely to overshoot, because a single noisy tree can swing the running prediction by a lot.

The trade-off in one sentence: **a smaller `learning_rate` needs more `n_estimators` to reach the same accuracy, but usually ends up at a *better* peak.** Slow, steady corrections smooth out individual-tree noise; loud, fast corrections tend to overcorrect. The sweep below holds `n_estimators=100` fixed and tries five learning rates (`0.01, 0.05, 0.1, 0.2, 0.5`) so the shape of the trade-off is visible on both business cases side-by-side. The State Health Department wants to know if dialing this knob meaningfully changes tumor-classification accuracy; HomeValue Analytics wants to know how much R² they leave on the table by accepting the default.


> 💡 **Gemini Prompt:** "Sweep learning_rate in [0.01, 0.05, 0.1, 0.2, 0.5] at fixed n_estimators=100 on both cases using 5-fold CV — cv_clf with scoring='roc_auc' for classification, cv_reg with scoring='r2' for regression. For each setting record the CV mean and the fold SD, print them as a table, and render a 1×2 plot of CV mean with SD error bars and a log-scaled x-axis for learning_rate."
>
> **After running, verify:**
> - [ ] Both panels use a log-scale x-axis
> - [ ] Classification scores stay inside a narrow band (about 0.97–0.99) across the sweep
> - [ ] Regression rises steeply from lr=0.01 (around R² 0.5) to the larger learning rates (around 0.8)
> - [ ] All `cross_val_score` calls use `n_jobs=-1` and 5-fold CV


In [ ]:
# Learning rate sweep — 5-fold CV at fixed n_estimators=100 (fold mean ± fold SD).
lr_grid = [0.01, 0.05, 0.1, 0.2, 0.5]

clf_lr_means, clf_lr_sds = [], []
reg_lr_means, reg_lr_sds = [], []
for lr in lr_grid:
    s_c = cross_val_score(GradientBoostingClassifier(n_estimators=100, learning_rate=lr, random_state=RANDOM_SEED),
                          X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1)
    clf_lr_means.append(s_c.mean()); clf_lr_sds.append(s_c.std(ddof=1))
    s_r = cross_val_score(GradientBoostingRegressor(n_estimators=100, learning_rate=lr, random_state=RANDOM_SEED),
                          X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1)
    reg_lr_means.append(s_r.mean()); reg_lr_sds.append(s_r.std(ddof=1))

print(pd.DataFrame({'learning_rate': lr_grid, 'clf ROC-AUC mean': clf_lr_means, 'clf SD': clf_lr_sds,
                    'reg R² mean': reg_lr_means, 'reg SD': reg_lr_sds}).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ax = axes[0]
ax.errorbar(lr_grid, clf_lr_means, yerr=clf_lr_sds, marker='o', linewidth=2, capsize=6, color=CLF_COLOR)
ax.set_xscale('log')
ax.set_xlabel('learning_rate (log scale)'); ax.set_ylabel('5-fold CV ROC-AUC (mean ± fold SD)')
ax.set_title('Classification — learning rate sweep at n=100', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.errorbar(lr_grid, reg_lr_means, yerr=reg_lr_sds, marker='o', linewidth=2, capsize=6, color=REG_COLOR)
ax.set_xscale('log')
ax.set_xlabel('learning_rate (log scale)'); ax.set_ylabel('5-fold CV R² (mean ± fold SD)')
ax.set_title('Regression — learning rate sweep at n=100', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

fig.suptitle('learning_rate × n_estimators are coupled — at 100 trees a tiny learning rate cannot fit',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

The two cases react to the learning-rate knob very differently — and that contrast is itself the lesson.

**Classification case — State Health Department (Wisconsin Breast Cancer).** Across the full sweep, CV ROC-AUC stays in a narrow **0.976–0.990** band. The slowest setting (`lr=0.01`) scores lowest (**0.976**, with the widest fold SD, 0.032) — 100 trees casting 1% votes each deliver only about one full tree's worth of correction. From `lr=0.1` upward the curve is nearly flat: **0.987**, **0.990** and **0.988** at 0.1, 0.2 and 0.5. **For the Health Department's classifier, the learning-rate knob barely moves the needle once it is large enough to fit.** Wisconsin is a near-ceiling problem (LogReg already sits at ROC-AUC 0.994), so there is little structure left for the boosting ensemble to learn however this knob is set.

**Regression case — HomeValue Analytics (California Housing).** Here the knob is consequential. At `lr=0.01` CV R² is only **0.505** — the ensemble has nowhere near enough total contribution to fit the structure in 16,512 census tracts. It climbs to **0.755** at `lr=0.05`, **0.790** at the default `lr=0.1`, **0.809** at `lr=0.2` and **0.814** at `lr=0.5` — a swing of about **31 R² points** from the slowest to the fastest setting at a fixed 100 trees. In business terms, that is the difference between a price model too coarse for neighborhood-level decisions and one that captures most of the price variance. At 100 trees the larger learning rates have not yet started to overshoot on this data; to reach the same accuracy with a smaller learning rate, the analyst has to add trees — which is exactly what Section 6's joint grid measures.

> **A question that often comes up here:** *"Why is classification so flat while regression cares so much about learning rate?"* Two reasons. First, **ceiling effects.** Wisconsin is already near the top of the score; there is barely any residual signal left for any model to extract, so the knob looks nearly identical at every usable setting. California Housing has rich non-linear structure that takes serious `learning_rate × n_estimators` budget to capture. Second, **what the loss function rewards.** Log-loss flattens out fast once the model is confidently right; squared error keeps rewarding tighter price fits all the way down. Tuning matters more wherever the loss curve still has room to fall.

**Key takeaway.** `learning_rate` and `n_estimators` come as a pair — never tune one without the other. On forgiving classification problems (near-ceiling tasks like Wisconsin), defaults are close to the top. On rich regression problems (the HomeValue case), the choice drives a swing of about 31 R² points at a fixed tree count. Section 6's joint grid is the right way to pick both at once.

---


## 6. Joint Tuning Grid — `n_estimators × learning_rate × max_features × max_depth`

Section 5 showed that `learning_rate` alone is not the whole story — its effect depends on how many trees the ensemble builds and on how complex each tree is. The right tool is a **joint grid** that tunes the four GBM knobs together:

- **`n_estimators ∈ {50, 100, 200}`** — three tree counts.
- **`learning_rate ∈ {0.05, 0.1, 0.2}`** — three learning rates.
- **`max_features ∈ {'sqrt', 0.5, None}`** — three feature-subset sizes; `None` (every feature at every split) is the GBM default.
- **`max_depth ∈ {3, None}`** — the GBM default of shallow trees, and unlimited depth (the forest's setting) to see whether bagging's deep trees transfer to boosting.

On **classification** that is 3 × 3 × 3 × 2 = **54 cells**. On **regression** the grid keeps only the **27 `max_depth=3` cells**: unlimited-depth boosting on 16,512 rows is by far the slowest corner of the grid on free Colab, and deep boosted trees are rarely competitive — so the *search* shrinks to fit the time budget. The **folds never shrink**: every cell is still a 5-fold estimate on `cv_reg`, so any cell can face any other model fold by fold. Every cell keeps its five fold scores, and the figure shows nine panels — rows are `max_features`, columns are classification depth 3, classification unlimited depth, and regression depth 3 — each a 3×3 heatmap of `(learning_rate × n_estimators)` with the **CV mean** and the **fold SD** in every cell.

As in nb12, the cell with the largest mean is only a **nominee**, and its score is optimistic: it was picked out of 54 (or 27) candidates on the same five folds that scored it. The nominee faces the **family default** — `GradientBoostingClassifier()` / `GradientBoostingRegressor()`, which is one of the grid's cells (`n=100`, `lr=0.1`, all features, depth 3) — with `compare_paired`, and replaces it only on a worthwhile gain. The GBM that comes out of this section is the one §7 puts on the ladder and nb14 inherits.


> 💡 **Gemini Prompt:** "Run a joint grid sweep with 5-fold CV, keeping each cell's five fold scores in a dictionary. Classification: GradientBoostingClassifier(random_state=474) over n_estimators [50, 100, 200] × learning_rate [0.05, 0.1, 0.2] × max_features ['sqrt', 0.5, None] × max_depth [3, None] with cv_clf and scoring='roc_auc' (54 cells). Regression: GradientBoostingRegressor(random_state=474) over the same n_estimators, learning_rate and max_features values at max_depth=3 only, with cv_reg and scoring='r2' (27 cells). Time both loops. Render a 3×3 figure: rows are max_features; columns are [clf depth 3, clf depth None, reg depth 3]; each panel a 3×3 heatmap of (learning_rate × n_estimators) annotated with the mean and the fold SD. Mark each case's nominee (the largest mean) with a red rectangle. Then run show_comparison(nominee = A, default GBM = B) per case — the default is the cell (depth 3, max_features None, lr 0.1, n 100) — and carry the nominee forward only if the verdict is 'A is better by a worthwhile amount'."
>
> **After running, verify:**
> - [ ] Nine panels arranged 3 × 3, each 3 rows (`learning_rate`) × 3 columns (`n_estimators`)
> - [ ] Each cell shows the CV mean and the fold SD — no ± interval
> - [ ] The run time of each grid is printed
> - [ ] One red rectangle per case, the nominee labelled optimistic, and a `compare_paired` verdict against the default GBM
> - [ ] The GBM carried forward is printed for each case


In [ ]:
# Joint 4D grid — n_estimators × learning_rate × max_features × max_depth, 5-fold CV, fold scores kept.
# Classification: all 54 cells. Regression: the 27 max_depth=3 cells only — unlimited-depth boosting on
# 16,512 rows is the slow corner of the grid on free Colab, so the SEARCH shrinks; the folds stay at five.
from matplotlib.patches import Rectangle
import time

n_grid  = [50, 100, 200]
lr_grid_joint = [0.05, 0.1, 0.2]
mf_grid_gbm = ['sqrt', 0.5, None]
mf_labels = ["'sqrt'", '0.5', 'None']

t0 = time.time()
grid_folds_clf = {}
for md in [3, None]:
    for mf in mf_grid_gbm:
        for lr in lr_grid_joint:
            for n in n_grid:
                grid_folds_clf[(md, mf, lr, n)] = cross_val_score(
                    GradientBoostingClassifier(n_estimators=n, learning_rate=lr, max_features=mf, max_depth=md, random_state=RANDOM_SEED),
                    X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1)
t_clf = time.time() - t0

t0 = time.time()
grid_folds_reg = {}
for mf in mf_grid_gbm:
    for lr in lr_grid_joint:
        for n in n_grid:
            grid_folds_reg[(3, mf, lr, n)] = cross_val_score(
                GradientBoostingRegressor(n_estimators=n, learning_rate=lr, max_features=mf, max_depth=3, random_state=RANDOM_SEED),
                X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1)
t_reg = time.time() - t0
print(f"Grid run time: classification {len(grid_folds_clf)} cells in {t_clf:.0f} s; regression {len(grid_folds_reg)} cells in {t_reg:.0f} s")

# Panels: rows = max_features; columns = [clf depth 3, clf depth None, reg depth 3]; cells = (learning_rate × n_estimators)
panels = [('clf', 3, 'Blues', 'Clf'), ('clf', None, 'Blues', 'Clf'), ('reg', 3, 'Oranges', 'Reg')]
folds_of = {'clf': grid_folds_clf, 'reg': grid_folds_reg}
nominee = {c: max(f, key=lambda key: f[key].mean()) for c, f in folds_of.items()}
lims = {c: (min(v.mean() for v in f.values()), max(v.mean() for v in f.values())) for c, f in folds_of.items()}

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for f_i, mf in enumerate(mf_grid_gbm):
    for col, (case, md, cmap, tag) in enumerate(panels):
        ax = axes[f_i, col]
        mean = np.array([[folds_of[case][(md, mf, lr, n)].mean() for n in n_grid] for lr in lr_grid_joint])
        sd   = np.array([[folds_of[case][(md, mf, lr, n)].std(ddof=1) for n in n_grid] for lr in lr_grid_joint])
        vmin, vmax = lims[case]
        ax.imshow(mean, cmap=cmap, aspect='auto', vmin=vmin, vmax=vmax)
        for i in range(3):
            for j in range(3):
                ax.text(j, i, f'{mean[i, j]:.4f}\nSD {sd[i, j]:.4f}', ha='center', va='center', fontsize=8,
                        color='black' if mean[i, j] < vmin + 0.8 * (vmax - vmin) else 'white', fontweight='bold')
        nom = nominee[case]
        if nom[0] == md and nom[1] == mf:
            ax.add_patch(Rectangle((n_grid.index(nom[3]) - 0.5, lr_grid_joint.index(nom[2]) - 0.5), 1, 1,
                                   fill=False, edgecolor='red', linewidth=3))
        ax.set_xticks(range(3)); ax.set_xticklabels([f'n={n}' for n in n_grid])
        ax.set_yticks(range(3)); ax.set_yticklabels([f'lr={lr}' for lr in lr_grid_joint])
        ax.set_title(f"{tag} — mf={mf_labels[f_i]}, md={md}", fontsize=10, fontweight='bold')
fig.suptitle('Joint GBM grid — CV mean and fold SD per cell; red box = the nominee (largest mean) per case',
             fontsize=13, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("CV-mean range per panel:")
for case, md, _, tag in panels:
    vals = [v.mean() for k, v in folds_of[case].items() if k[0] == md]
    print(f"  {tag} max_depth={md}: {min(vals):.4f}–{max(vals):.4f}")

# --- D3: the nominee faces the family default. GradientBoosting's defaults (n=100, lr=0.1, all features,
# depth 3) are one of the cells, so the default's fold scores come straight from the grid.
def _cfg(key):
    md, mf, lr, n = key
    return f"n_estimators={n}, learning_rate={lr}, max_features={mf!r}, max_depth={md}"
default_key = (3, None, 0.1, 100)
gbm_final, gbm_label = {}, {}
for case, Est, deltas, metric in (('clf', GradientBoostingClassifier, DELTAS_AUC, 'ROC-AUC'),
                                  ('reg', GradientBoostingRegressor, DELTAS_R2, 'R²')):
    nom = nominee[case]
    print(f"\nNominee — {case}: {_cfg(nom)} → mean {folds_of[case][nom].mean():.4f} "
          f"(optimistic: picked from {len(folds_of[case])} cells on these folds)")
    _, _, _, v = show_comparison('Grid nominee', 'Default GBM', folds_of[case][nom], folds_of[case][default_key], deltas, metric)
    if v.startswith('A is better'):
        md, mf, lr, n = nom
        gbm_final[case] = Est(n_estimators=n, learning_rate=lr, max_features=mf, max_depth=md, random_state=RANDOM_SEED)
        gbm_label[case] = f'GBM tuned ({_cfg(nom)})'
    else:
        gbm_final[case] = Est(random_state=RANDOM_SEED)
        gbm_label[case] = 'GBM (default)'
print()
print(f"✓ GBM carried forward — classification: {gbm_label['clf']}")
print(f"✓ GBM carried forward — regression:     {gbm_label['reg']}")


**Reading the output:**

The nine-panel figure shows the grid; the printed lines below it give each panel's range and the comparisons that decide. (On this machine the classification grid took 11 s and the regression grid 52 s; on free Colab expect a few minutes for the regression grid.)

**Classification case — State Health Department.** The depth-3 panels span **0.979–0.993** ROC-AUC; the unlimited-depth panels span **0.938–0.989** — the best deep-tree cell stays below the best shallow one, and the worst falls well behind. The nominee is `(n_estimators=200, learning_rate=0.1, max_features='sqrt', max_depth=3)` at **0.9931**. Against the default GBM (0.9872) its per-fold gains are all positive, +0.002 to +0.014, a mean of **+0.0059** — but the corrected interval is **[−0.004, +0.016]**: ***unresolved***. The default GBM stays as the classification GBM, and no margin in the band would change that.

**Regression case — HomeValue Analytics.** The depth-3 grid spans **0.622–0.822** R², and the structure is the `(learning_rate × n_estimators)` budget from Section 5: more total correction, higher score, all the way to the grid's corner. The nominee is `(n_estimators=200, learning_rate=0.2, max_features=None, max_depth=3)` at **0.8217** — that is, scikit-learn's default GBM with only the tree count and the learning rate raised. Against the default GBM (0.7900) its per-fold gains run from **+0.024 to +0.043**, mean **+0.0317**, corrected interval **[+0.017, +0.047]**: ***A is better by a worthwhile amount*** at the base margin (and at the small one; at the large margin of 0.02 it would be unresolved). **HomeValue's GBM is the tuned one**: `GradientBoostingRegressor(n_estimators=200, learning_rate=0.2, random_state=474)`. Its 0.8217 is optimistic — it is the best of 27 cells on these folds — and §7 now puts it against the forest, the actual champion.

> **A question that often comes up here:** *"Why does unlimited depth help a forest but not a boosted model?"* Because **bagging and boosting handle deep trees very differently**. Bagging averages predictions from independent deep trees fit on bootstrap samples — each tree may overfit individually, but averaging cancels much of that variance. Boosting fits each tree on the residuals of the *running ensemble*: a deep tree at iteration *t* fits the noise left over from iterations 1…*t*−1, iteration *t*+1 fits the residual of *that* fit, and the errors compound. The classic GBM recipe is *shallow trees + many iterations + a moderate learning rate* — on the classification grid the depth-3 cells hold the top, and on regression the grid does not even spend time on unlimited depth.

**Key takeaway.** Tune the four GBM knobs together, keep every cell's fold scores, and let the nominee face the default GBM on the same folds. On **classification** the nominee is unresolved against the default, so the **default GBM** goes forward. On **regression** the nominee — the default with `n_estimators=200` and `learning_rate=0.2` — earns a worthwhile gain of about 0.03 R², so the **tuned GBM** goes forward to face the forest in §7.

---


## 📝 PAUSE-AND-DO Exercise 1 (clf, 5 minutes) — Tune the Classification GBM

**Task:** Find the best `(n_estimators, learning_rate)` combination for `GradientBoostingClassifier` on Wisconsin breast cancer with a 3×3 grid, and test whether it earns a place over the default GBM.

**Instructions:**
1. Sweep `n_estimators ∈ [50, 100, 200]` × `learning_rate ∈ [0.05, 0.1, 0.2]` (other settings at their defaults).
2. For each combination, compute the 5-fold CV ROC-AUC mean and SD using `cv_clf`, keeping each cell's five fold scores.
3. Render the means as a heatmap with values annotated and the best cell starred.
4. Treat the best cell as a **nominee** and compare it with the default GBM (`n_estimators=100`, `learning_rate=0.1` — one of your nine cells) using `compare_paired(nominee_scores, default_scores, DELTA_AUC)`.
5. Write 3 short findings: does the learning-rate × tree-count pattern from Sections 5 and 6 show up, and did the nominee earn its place?

---


> 💡 **Gemini Prompt:** "Grid-search GradientBoostingClassifier(random_state=474) over n_estimators=[50,100,200] × learning_rate=[0.05,0.1,0.2] using cross_val_score with cv_clf and scoring='roc_auc' on X_train_clf, keeping each cell's five fold scores. Print tables of the means and SDs, draw a 3×3 heatmap with the best cell starred, and run show_comparison with the best cell as A and the (n=100, lr=0.1) cell — the default GBM — as B, using DELTAS_AUC."
>
> **After running, verify:**
> - [ ] Heatmap has 9 cells with mean values annotated and one star
> - [ ] The nominee is printed and labelled optimistic
> - [ ] `compare_paired` output shows per-fold gains, a corrected interval and one of the four verdicts


In [ ]:
# YOUR SOLUTION CODE HERE
# Grid over n_estimators × learning_rate with 5-fold CV ROC-AUC (cv_clf), keeping each cell's fold scores.
# The best cell is a nominee: compare it with the default GBM (n=100, lr=0.1) using compare_paired.


## 📝 PAUSE-AND-DO Exercise 2 (reg, 5 minutes) — Tune the Regression GBM

**Task:** Find the best `(n_estimators, learning_rate)` combination for `GradientBoostingRegressor` on California Housing, and test whether it earns a place over the default GBM.

**Instructions:**
1. Same 3×3 grid as Exercise 1 (`n ∈ [50, 100, 200]` × `lr ∈ [0.05, 0.1, 0.2]`, other settings at their defaults).
2. 5-fold CV R² with `cv_reg`, keeping each cell's five fold scores.
3. Heatmap with mean annotations and the best cell starred.
4. Compare the nominee with the default GBM (`n=100, lr=0.1`) using `compare_paired(..., DELTA_R2)`, and report both models' CV RMSE in USD.
5. Write 3 short findings: how does this grid differ from Exercise 1's, and what does the verdict mean for HomeValue?

---


> 💡 **Gemini Prompt:** "Grid-search GradientBoostingRegressor(random_state=474) over n_estimators=[50,100,200] × learning_rate=[0.05,0.1,0.2] using cross_val_score with cv_reg and scoring='r2' on X_train_reg, keeping each cell's five fold scores. Print a table of the means, draw a 3×3 heatmap with the best cell starred, run show_comparison with the best cell as A and the (n=100, lr=0.1) default as B using DELTAS_R2, and print the CV RMSE in USD (multiply by 100,000) of both models."
>
> **After running, verify:**
> - [ ] Heatmap has 9 cells with mean values annotated and one star
> - [ ] `compare_paired` output shows per-fold gains, a corrected interval and one of the four verdicts
> - [ ] CV RMSE printed in USD for the nominee and the default


In [ ]:
# YOUR SOLUTION CODE HERE
# Same n_estimators × learning_rate grid with 5-fold CV R² (cv_reg); compare the nominee with the default GBM.


## 7. Final Comparison — The Champion Ladder on Both Cases

The closing section is the **direct setup for nb14's selection ceremony**. Four candidates per business case, scored on identical 5-fold folds and climbed as a **ladder** in order of complexity — each one a model the course has already earned its way to:

1. **nb09 reference (the incumbent)** — `LogReg(C=1.0)` for classification, OLS for regression.
2. **nb11's single tree** — `DecisionTreeClassifier(max_depth=3)` / `DecisionTreeRegressor(max_depth=5)`, the readable trees.
3. **nb12's random forest** — the default forest on both cases (neither grid nominee beat it in nb12).
4. **This notebook's GBM** — the default GBM on classification (the nominee was unresolved against it), the tuned GBM `(n_estimators=200, learning_rate=0.2)` on regression (it earned a worthwhile gain over the default).

`champion = incumbent`; for each challenger in order, `compare_paired(challenger, champion, δ)` on the same folds; promote only on *A is better by a worthwhile amount*. *Unresolved* keeps the current champion (the stakeholders' declared default), and *practically equivalent* keeps the simpler model, which on this ladder is always the current champion. Then two checks from nb08's band rules: re-run the whole ladder at the small and large margins — the champion is **robust** if it is the same at all three, **margin-sensitive** otherwise — and report the **switching cutoff** for the closest rival (the unpromoted challenger whose interval came closest to a worthwhile gain), read straight off its interval.


> 💡 **Gemini Prompt:** "Score four candidates per case on identical folds: classification — reference_clf, DecisionTreeClassifier(max_depth=3, random_state=474), RandomForestClassifier(random_state=474) and the GBM carried from Section 6, with cv_clf and scoring='roc_auc'; regression — reference_reg, DecisionTreeRegressor(max_depth=5, random_state=474), RandomForestRegressor(random_state=474) and the GBM carried from Section 6, with cv_reg and scoring='r2'. Print each case's fold_table. Write run_ladder(scores, ladder, margin): champion = ladder[0]; for each challenger, compare_paired(challenger, champion, margin) and promote only on 'A is better by a worthwhile amount'. Run it at the base margin with show_comparison printing every step, then silently at the small and large margins and report whether the champion is robust or margin-sensitive. For the unpromoted challenger with the highest interval lower limit, print the switching cutoff. Plot the fold scores and every ladder step's corrected interval against ±δ."
>
> **After running, verify:**
> - [ ] Every candidate on a case uses the same `cv_clf` / `cv_reg` object
> - [ ] Each ladder step prints per-fold gains, the corrected interval, the verdict and the champion after the step
> - [ ] The champion is printed at the small, base and large margins, with "robust" or "margin-sensitive"
> - [ ] The closest rival's switching cutoff is printed


In [ ]:
# Champion ladder per case — incumbent first, then challengers in order of complexity.
# Every candidate runs on the same splitter (cv_clf / cv_reg), so every step is a paired comparison.
clf_compare = {
    'nb09 reference: LogReg(C=1.0)': cross_val_score(reference_clf, X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
    'Single Tree (depth=3, nb11)':   cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED), X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
    'Random Forest (default, nb12)': cross_val_score(forest_clf_carried, X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
    gbm_label['clf']:                cross_val_score(gbm_final['clf'], X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
}
reg_compare = {
    'nb09 reference: OLS':           cross_val_score(reference_reg, X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1),
    'Single Tree (depth=5, nb11)':   cross_val_score(DecisionTreeRegressor(max_depth=5, random_state=RANDOM_SEED), X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1),
    'Random Forest (default, nb12)': cross_val_score(forest_reg_carried, X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1),
    gbm_label['reg']:                cross_val_score(gbm_final['reg'], X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1),
}

print("=== CLASSIFICATION — 5-fold CV ROC-AUC: fold scores, mean, SD ===")
print(fold_table(clf_compare).to_string(index=False))
print()
print("=== REGRESSION — 5-fold CV R²: fold scores, mean, SD ===")
print(fold_table(reg_compare).to_string(index=False))

def run_ladder(scores, ladder, margin, deltas=None, metric='', verbose=True):
    """Incumbent first; a challenger replaces the current champion only on a worthwhile gain at `margin`."""
    champion, log = ladder[0], []
    for challenger in ladder[1:]:
        if verbose:
            print()
            d, gain, ci, verdict = show_comparison(challenger, champion, scores[challenger], scores[champion], deltas, metric)
        else:
            d, gain, ci, verdict = compare_paired(scores[challenger], scores[champion], margin)
        log.append({'challenger': challenger, 'vs': champion, 'd': d, 'ci': ci, 'verdict': verdict,
                    'promoted': verdict.startswith('A is better')})
        if log[-1]['promoted']:
            champion = challenger
        if verbose:
            print(f"  → champion now: {champion}")
    return champion, log

results = {}
for case, scores, deltas, metric in (('MedScreen', clf_compare, DELTAS_AUC, 'ROC-AUC'),
                                     ('HomeValue', reg_compare, DELTAS_R2, 'R²')):
    print(f"\n######## {case} ladder ({metric}, base δ = {deltas['base']}) ########")
    champ, log = run_ladder(scores, list(scores), deltas['base'], deltas, metric)
    band = {name: run_ladder(scores, list(scores), m, verbose=False)[0] for name, m in deltas.items()}
    print(f"\n{case} champion at base δ: {champ}")
    print("  band: " + '; '.join(f"{n} {deltas[n]} → {c}" for n, c in band.items())
          + ("  → robust across the band" if len(set(band.values())) == 1 else "  → margin-sensitive"))
    rivals = [e for e in log if not e['promoted']]
    if rivals:
        r = max(rivals, key=lambda e: e['ci'][0])
        lo, hi = r['ci']
        gain_part = (f"a worthwhile gain only for δ below {lo:.4f}" if lo > 0
                     else "no positive δ makes it a worthwhile gain")
        print(f"  closest rival: {r['challenger']} vs {r['vs']}, interval [{lo:+.4f}, {hi:+.4f}] → "
              f"{gain_part}; practically equivalent only for δ above {max(abs(lo), abs(hi)):.4f}")
    results[case] = (champ, log)

fig, axes = plt.subplots(2, 2, figsize=(18, 11))
plot_fold_scores(axes[0, 0], clf_compare, '5-fold CV ROC-AUC', 'MedScreen — fold scores (bar = mean)')
plot_fold_scores(axes[0, 1], reg_compare, '5-fold CV R²', 'HomeValue — fold scores (bar = mean)')
for ax, case, margin, unit in ((axes[1, 0], 'MedScreen', DELTA_AUC, 'ROC-AUC'), (axes[1, 1], 'HomeValue', DELTA_R2, 'R²')):
    rows = [(f"{e['challenger'][:28]} − {e['vs'][:22]}", e['d'], e['ci']) for e in results[case][1]]
    plot_gain_intervals(ax, rows, margin, f'Gain in {unit} (challenger − champion)', f'{case} ladder — each step vs ±δ')
fig.suptitle('The champion ladder nb14 inherits — a challenger is promoted only on a worthwhile gain',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


**Reading the output:**

Two business cases, two ladders, two champions — and one of them comes with an honest caveat.

**Classification case (Wisconsin Breast Cancer — State Health Department).** LogReg(C=1.0) averages **0.994** ROC-AUC, the tree **0.940**, the default forest **0.988** and the default GBM **0.987**. Every challenger is ***unresolved*** against LogReg: the tree [−0.143, +0.035], the forest [−0.019, +0.007], the default GBM [−0.021, +0.007]. None is promoted, at any margin in the band — the champion is **robust**. The closest rival is the forest (highest lower limit, −0.019); because that limit is below zero, no positive margin would make it a worthwhile gain, and it would be practically equivalent only for a margin above 0.019. **Champion: `LogReg(C=1.0)`** — and, as in nb12, part of the reason is structural: with two folds already at 0.9995, this evidence cannot certify any ranking gain above about 0.0005, so the explainable model stays because nothing better can be shown, not because it has been proven best.

**Regression case (California Housing — HomeValue Analytics).** OLS averages **0.604** R², the tree **0.611**, the default forest **0.804** and the tuned GBM **0.822**. *Tree vs OLS:* unresolved, OLS stays. *Forest vs OLS:* mean gain **+0.199**, corrected interval **[+0.180, +0.218]** — a worthwhile gain; the forest becomes champion. *Tuned GBM vs forest:* the GBM leads on all five folds, by **+0.011 to +0.023**, a mean gain of **+0.018** — but the corrected interval is **[+0.009, +0.027]**, and its lower end sits just below the 0.01 margin. That is ***unresolved***, so the declared default applies: **the forest stays champion**. The band shows how close this is: at the small margin (0.005) the GBM would be promoted, at the base and large margins the forest stays — the champion is **margin-sensitive**. The switching cutoff says exactly where the line sits: the GBM is a worthwhile gain over the forest only for a margin below **0.0091** R², and practically equivalent only above **0.0271**.

The champion statement for HomeValue's CFO, in the four parts nb14 will formalize:

> **Champion.** We recommend the **random forest** (`RandomForestRegressor(random_state=474)`) for HomeValue's neighborhood value estimates. **Evidence.** Against OLS, the model HomeValue runs today, on the same five folds, it is better by a worthwhile amount: mean gain **+0.199 R²**, interval corrected for fold overlap **[+0.180, +0.218]**, against the **δ = 0.01** set before comparing. **Checks.** The tuned GBM scored higher on average, but the evidence does not show a gain large enough to justify switching to it, so under the CFO's policy we keep the forest; this verdict would flip for a CFO willing to switch for a gain as small as 0.005 R². **What would change our mind.** The tuned GBM is the candidate for a future evaluation — not an automatic fallback; we revisit if the CFO lowers the margin or if new data narrows the interval.

> **A question that often comes up here:** *"The GBM beat the forest on every fold and has the higher mean. Why isn't it the champion?"* Because neither of those is the test. Five fold wins are description: with no real difference, five same-direction folds happen by chance more often than you would think, especially since CV folds share training rows. The test is the corrected interval of the gain against the margin the CFO set *before* seeing these numbers — and here its lower end (0.009) does not clear 0.01. Two more cautions point the same way: the tuned GBM was itself picked from a 27-cell grid on these folds, so its score is optimistic, while the forest is an untuned default. Reporting "margin-sensitive" and the switching cutoff is what keeps this honest — the CFO can see that a slightly more relaxed policy would favor the GBM, but that policy has to be declared in advance, not chosen after the scores are in.

**Key takeaway.** **MedScreen: `LogReg(C=1.0)`** — every challenger unresolved, robust across the band. **HomeValue: the random forest** — a worthwhile gain over OLS; the tuned GBM's edge over the forest is unresolved at the base margin, so the forest stays, and the verdict is margin-sensitive (switching cutoff 0.0091 R²). These are the rosters and verdicts nb14's selection ceremony inherits.

---


## 8. Wrap-Up — Key Takeaways

**What landed today:**

1. **Boosting is sequential, not parallel.** Each tree fits the residuals of the running ensemble — on regression the literal `y − ŷ`, on classification the gradient of the log-loss, which works out to roughly `y − p̂`. The algorithm reduces **bias** (systematic mistakes) rather than variance.

2. **`learning_rate` and `n_estimators` are coupled.** Tune them together. On California Housing the equal-budget diagonal of the grid lands on almost the same R² (about 0.79), and the score rises with the total budget up to the grid's corner (0.822 at `n=200, lr=0.2`); on near-ceiling Wisconsin the `n_estimators × learning_rate` grid of Exercise 1 spans only about 0.01 ROC-AUC.

3. **A tuning grid nominates; the default decides whether the nominee is needed.** On classification the GBM nominee was unresolved against the default GBM, so the default goes forward. On regression the nominee — the default with `n_estimators=200` and `learning_rate=0.2` — earned a worthwhile gain of about 0.03 R², so the tuned GBM goes forward. Either way the nominee's own score is optimistic, because it was picked on the folds that scored it.

4. **Per-case champions — climbed as a ladder, checked across the band.**
   - **Classification (Wisconsin Breast Cancer — State Health Department):** **`LogReg(C=1.0)`**. The tree, the default forest and the default GBM are all *unresolved* against it, at every margin in the band.
   - **Regression (California Housing — HomeValue Analytics):** **the random forest** (`RandomForestRegressor(random_state=474)`). It earned a worthwhile gain over OLS (+0.199 R², [+0.180, +0.218]); the tuned GBM's gain over the forest (+0.018, [+0.009, +0.027]) is *unresolved* at the base margin, so the forest stays. The verdict is **margin-sensitive** — the GBM would win at δ = 0.005 — and the switching cutoff is 0.0091 R².

5. **Boosting amplifies leaky features more aggressively than any algorithm in the course — defend against it.** Because every iteration fits the residuals, a leaky column (one engineered from future data, target data, or held-out data) drives the residuals to zero in the first few trees, and the model becomes `y = leaky_feature` plus tiny corrections. On a held-out test set or in production where the leaky column is missing or noisy, the model collapses. The defence is the four-method importance check from nb12 §7: drop or null suspect features one at a time and refit on the same folds (a CV collapse means the feature was carrying signal it should not have); inspect the top-ranked feature in the four-method heatmap (derived fields with names like `_at_t+1`, `_after_`, `_total_lifetime_`, or anything engineered from the label are the usual suspects); and audit the pipeline boundary for any column engineered from future data, target data, or held-out data. The same recipe protects you on every tabular dataset; the urgency just goes up with boosting.

**Bridge to nb14 — the Selection Ceremony:**

You now have the candidate roster per business case — every model scored on identical 5-fold folds, climbed as a ladder — that nb14 turns into a formal selection ceremony with a written champion statement and a single, blind opening of the locked test set.

- **Classification:** nb09 LogReg(C=1.0), single tree (depth 3), the default random forest, the default GBM. nb14 adds a regularized linear variant (LogReg L1).
- **Regression:** nb09 OLS, single tree (depth 5), the default random forest, the tuned GBM `(n_estimators=200, learning_rate=0.2)`. nb14 adds regularized linear variants.

nb14 does three things you have not yet seen as a single workflow: (a) **declare the protocol in writing before looking at any results** — the primary metric, the margin band, the incumbent and the challenger order (no model-shopping after seeing the numbers); (b) **write the champion statement** — champion, evidence (the corrected interval against δ), checks, and what would change your mind; (c) **freeze the champion and open the locked test set exactly once per business case**, judging the frozen champion against the deployment requirement declared in advance. The test score never re-picks the champion.

> **A question that often comes up at this point:** *"Does gradient boosting always beat random forest?"* Not on the evidence you produced today. On California Housing the tuned GBM averages about 0.018 R² more than the forest and leads on every fold, but the corrected interval of that gain, [+0.009, +0.027], does not clear the CFO's 0.01 margin — so the forest stays, with the GBM named as the candidate for a future evaluation. On Wisconsin breast cancer no tree-based candidate shows any worthwhile gain over the logistic reference. Boosting is a strong tool; whether it *ships* is always a paired comparison against the current champion, read against a margin declared before the scores were computed.

---


## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete both PAUSE-AND-DO exercises** — Exercise 1 (GBM tuning, classification) and Exercise 2 (GBM tuning, regression).
2. **Run All Cells** — `Runtime → Run all` to ensure every cell executes without error.
3. **Save a Copy** — `File → Save a copy in Drive`, or download as `.ipynb`.
4. **Submit** — upload the `.ipynb` file to the Notebook 13 participation assignment on Brightspace.

### Before Submitting, Check:

- [ ] Both exercise solutions produce a tuning heatmap with the nominee starred and a `compare_paired` verdict against the default GBM
- [ ] The Section 7 champion ladder (fold-score strips, every step's corrected interval, the band check) renders for both cases
- [ ] All figures render (none broken)

### Next Step:

- **Notebook 14** — Model Selection Protocol + Test-Set Opening Ceremony

---

---

## Bibliography

- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2023). *An Introduction to Statistical Learning with Python* — Ch. 8 (Tree-Based Methods), §8.2.3 Boosting. Springer.
- Friedman, J. H. (2001). "Greedy function approximation: A gradient boosting machine." *Annals of Statistics*, 29(5), 1189–1232.
- Friedman, J. H. (2002). "Stochastic gradient boosting." *Computational Statistics & Data Analysis*, 38(4), 367–378.
- Chen, T., & Guestrin, C. (2016). "XGBoost: A scalable tree boosting system." *Proceedings of KDD '16*.
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* — Ch. 10: Boosting and Additive Trees.
- scikit-learn User Guide — [Gradient Tree Boosting](https://scikit-learn.org/stable/modules/ensemble.html#gradient-boosting)
- Nadeau, C., & Bengio, Y. (2003). "Inference for the generalization error." *Machine Learning*, 52, 239–281. (The corrected interval in `compare_paired`.)
- Cawley, G. C., & Talbot, N. L. C. (2010). "On over-fitting in model selection and subsequent selection bias in performance evaluation." *Journal of Machine Learning Research*, 11, 2079–2107. (Why a grid's best cell is optimistic.)

---

<center>

**Thank you!**

</center>